In [ ]:

# Step 1: Preprocess audio -> log-mel spectrograms

from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
from tqdm import tqdm

DATA_DIR = Path("../data")
OUT_DIR = Path("../npy")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TSV_PATH = DATA_DIR / "validated.tsv"
CLIPS_DIR = DATA_DIR / "clips"

USE_ALL = True
N_PARTIAL = 8000
RANDOM_SEED = 123

# Audio / feature settings
SR = 16000
N_MELS = 64
N_FFT = 400
HOP_LENGTH = 128
TOP_DB = 80.0

MAX_SEC = 5.0

# Store the log-mel spectrograms as float32.
DTYPE = np.float32
PAD_VALUE = -TOP_DB


In [ ]:

# Load validated.tsv

df = pd.read_csv(TSV_PATH, sep="\t", low_memory=False)
df["source_index"] = df.index.astype(np.int64)

print("validated rows:", len(df))
print(df.columns.tolist())
display(df.head())


In [ ]:

# Select samples

if USE_ALL:
    df_use = df.copy()
else:
    n = min(N_PARTIAL, len(df))
    df_use = (
        df.sample(n=n, random_state=RANDOM_SEED)
          .sort_values("source_index")
          .copy()
    )

print("selected rows:", len(df_use))


In [ ]:

# Keep clips up to MAX_SEC seconds.

keep_rows = []

for row_idx, row in tqdm(df_use.iterrows(), total=len(df_use)):
    mp3_path = CLIPS_DIR / row["path"]
    duration = float(librosa.get_duration(path=mp3_path))

    if duration <= MAX_SEC:
        keep_rows.append(row_idx)

df_use = df_use.loc[keep_rows].reset_index(drop=True)

print("usable rows:", len(df_use))


In [ ]:

# Determine X.npy shape.
#
# With the current settings:
#   SR=16000, MAX_SEC=5.0, HOP_LENGTH=128
# gives T_MAX = 626.

if MAX_SEC is None:
    max_sec = 0.0
    for _, row in tqdm(df_use.iterrows(), total=len(df_use)):
        duration = get_duration_sec(CLIPS_DIR / row["path"])
        max_sec = max(max_sec, duration)
else:
    max_sec = float(MAX_SEC)

MAX_SAMPLES = int(np.ceil(max_sec * SR))
T_MAX = 1 + MAX_SAMPLES // HOP_LENGTH

print("max_sec:", max_sec)
print("N_MELS:", N_MELS)
print("HOP_LENGTH:", HOP_LENGTH)
print("T_MAX:", T_MAX)
print("X shape:", (len(df_use), 1, N_MELS, T_MAX))


In [ ]:

# Convert MP3 -> log-mel and save X.npy / meta.csv

N = len(df_use)

X = np.lib.format.open_memmap(
    OUT_DIR / "X.npy",
    mode="w+",
    dtype=DTYPE,
    shape=(N, 1, N_MELS, T_MAX),
)

meta_rows = []

for i, row in tqdm(df_use.iterrows(), total=N):
    mp3_path = CLIPS_DIR / row["path"]

    y, _ = librosa.load(
        mp3_path,
        sr=SR,
        mono=True,
    )

    if len(y) > MAX_SAMPLES:
        raise RuntimeError(f"Audio exceeds MAX_SEC after filtering: {row['path']}")

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        power=2.0,
    )

    logmel = librosa.power_to_db(
        mel,
        ref=np.max,
        top_db=TOP_DB,
    )

    T = logmel.shape[1]

    if T > T_MAX:
        raise RuntimeError(f"T > T_MAX: {row['path']}")

    X[i, 0, :, :] = PAD_VALUE
    X[i, 0, :, :T] = logmel.astype(DTYPE)

    meta_rows.append({
        "index": i,
        "sentence": row["sentence"],
        "gender": row["gender"],
        "valid_max_index": T - 1,
        "source_index": int(row["source_index"]),
    })

X.flush()
del X

meta = pd.DataFrame(
    meta_rows,
    columns=[
        "index",
        "sentence",
        "gender",
        "valid_max_index",
        "source_index",
    ],
)

meta.to_csv(
    OUT_DIR / "meta.csv",
    index=False,
    encoding="utf-8-sig",
)

print("saved:", OUT_DIR / "X.npy")
print("saved:", OUT_DIR / "meta.csv")
print("X shape:", (N, 1, N_MELS, T_MAX))
display(meta.head())


In [ ]:

# Quick check

X = np.load(OUT_DIR / "X.npy", mmap_mode="r")
meta = pd.read_csv(OUT_DIR / "meta.csv")

print("X.shape =", X.shape)
print("meta.shape =", meta.shape)

i = 0
vmax = int(meta.loc[i, "valid_max_index"])

print("sentence:", meta.loc[i, "sentence"])
print("gender:", meta.loc[i, "gender"])
print("valid_max_index:", vmax)
print("source_index:", meta.loc[i, "source_index"])

plt.figure(figsize=(10, 4))
plt.imshow(
    X[i, 0, :, :vmax + 1],
    origin="lower",
    aspect="auto",
)
plt.xlabel("Time frame")
plt.ylabel("Mel bin")
plt.tight_layout()
plt.show()


In [ ]:

# Check correspondence with validated.tsv

source_index = int(meta.loc[0, "source_index"])

print("meta:")
print(meta.loc[0, "sentence"])

print("\nvalidated.tsv:")
print(df.iloc[source_index]["sentence"])
